### Assignment-03

- Using the provided ```U-Net``` training notebook as a reference, develop and train a ```DeepLabv3+``` model on the sample dataset available in the ```data``` folder (accessible through the provided Google Drive shortcut).

- For model evaluation, use the same test image that was previously used to evaluate the U-Net model. Save and upload the output image generated by your trained DeepLabv3+ model to the ```data/output``` folder.

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
import torchvision.transforms as transforms
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Constants (Matching your notebook's configuration)
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.0001 # Slightly lower for DeepLab transfer learning
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Reuse your SimpleDataset class
# Simple config
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimpleDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),  # forces uniform size
            transforms.ToTensor()
        ])

        imgs = list(self.images_dir.glob('*.jpg'))
        self.pairs = [img for img in imgs
                     if (self.masks_dir / f"{img.stem}_mask.png").exists()]
        print(f" Dataset ready: {len(self.pairs)} pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path = self.pairs[idx]
        mask_path = self.masks_dir / f"{img_path.stem}_mask.png"

        # Load image (BGR→RGB)
        img = cv2.imread(str(img_path))[...,::-1]

        # Load mask and normalize
        mask = cv2.imread(str(mask_path), 0)
        mask = (mask > 127).astype(np.float32)  # Binary 0/1

        # Transform RESIZES everything to 128x128
        img = self.transform(img)
        mask = self.transform(mask)[0][None]  # 1x128x128

        return img, mask

In [22]:
# 2. DeepLabv3+ Model Setup
def get_deeplab_model(output_channels=1):
    # Using ResNet-50 backbone
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')

    # Adjust the classifier for binary segmentation
    # DeepLabv3+ head consists of ASPP and a final conv layer
    model.classifier = DeepLabHead(2048, output_channels)

    return model.to(DEVICE)

In [26]:
# 3. Setup Training
dataset = SimpleDataset(
    '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/train2017',
    '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/mask_train2017'
)
train_loader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=2)

model = get_deeplab_model()
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

 Dataset ready: 3768 pairs


In [ ]:
import os
import shutil
from pathlib import Path

# Source paths from your notebook
DRIVE_IMG_DIR = '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/train2017'
DRIVE_MASK_DIR = '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/mask_train2017'

# Local destination
LOCAL_IMG_DIR = '/content/local_images'
LOCAL_MASK_DIR = '/content/local_masks'

def sync_data():
    # Clean up old attempts
    if os.path.exists(LOCAL_IMG_DIR): shutil.rmtree(LOCAL_IMG_DIR)
    if os.path.exists(LOCAL_MASK_DIR): shutil.rmtree(LOCAL_MASK_DIR)

    print("⏳ Copying images to local storage... (Faster than Drive)")
    shutil.copytree(DRIVE_IMG_DIR, LOCAL_IMG_DIR)
    shutil.copytree(DRIVE_MASK_DIR, LOCAL_MASK_DIR)

    # Verification
    img_count = len(list(Path(LOCAL_IMG_DIR).glob('*.jpg')))
    mask_count = len(list(Path(LOCAL_MASK_DIR).glob('*.png')))
    print(f"✅ Verified: {img_count} images and {mask_count} masks copied locally.")
    return img_count > 0

sync_data()

In [32]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

class FastDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_size=128):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size

        self.images = []
        self.masks = []

        # 1. Identify all valid pairs
        all_imgs = list(self.images_dir.glob('*.jpg'))

        print(f"⚡ Pre-loading {len(all_imgs)} images into RAM for maximum speed...")

        for img_path in tqdm(all_imgs):
            # Check both possible mask naming conventions
            mask_path = self.masks_dir / f"{img_path.stem}_mask.png"
            if not mask_path.exists():
                mask_path = self.masks_dir / f"{img_path.stem}.png"

            if mask_path.exists():
                # Load, RGB convert, and Resize Image
                img = cv2.imread(str(img_path))[...,::-1]
                img = cv2.resize(img, (img_size, img_size))

                # Load and Resize Mask
                mask = cv2.imread(str(mask_path), 0)
                mask = cv2.resize(mask, (img_size, img_size))
                mask = (mask > 127).astype(np.float32)

                # Store in lists (Caching)
                self.images.append(img)
                self.masks.append(mask)

        if len(self.images) == 0:
            raise ValueError(f"No pairs found in {images_dir}. Check folder paths!")

        print(f"✅ Successfully cached {len(self.images)} pairs in memory.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Pull directly from RAM (No Disk I/O)
        img = self.images[idx]
        mask = self.masks[idx]

        # Fast conversion to Tensors
        # Image: (H, W, C) uint8 -> (C, H, W) float32 [0, 1]
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        # Mask: (H, W) -> (1, H, W)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)

        return img_tensor, mask_tensor

# --- Usage with high-speed settings ---
IMAGE_PATH = '/content/local_images' # Local disk path
MASK_PATH = '/content/local_masks'

dataset = FastDataset(IMAGE_PATH, MASK_PATH, img_size=128)

# pin_memory=True speeds up data transfer to the GPU
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,   # Set to 0 when data is in RAM to avoid multi-processing overhead
    pin_memory=True
)

⚡ Pre-loading 11827 images into RAM for maximum speed...


100%|██████████| 11827/11827 [00:02<00:00, 4369.80it/s]

✅ Successfully cached 198 pairs in memory.


In [33]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from tqdm import tqdm

class FastDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_size=128):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.images = []
        self.masks = []

        # Support both .jpg and .JPG extensions
        all_imgs = list(self.images_dir.glob('*.jpg')) + list(self.images_dir.glob('*.JPG'))

        print(f"⚡ Pre-loading {len(all_imgs)} images into RAM for 10x speed...")
        for img_path in tqdm(all_imgs):
            # Check standard mask naming from your U-Net notebook
            mask_path = self.masks_dir / f"{img_path.stem}_mask.png"
            if not mask_path.exists():
                mask_path = self.masks_dir / f"{img_path.stem}.png"

            if mask_path.exists():
                # Load and Resize once
                img = cv2.imread(str(img_path))[...,::-1]
                img = cv2.resize(img, (img_size, img_size))

                mask = cv2.imread(str(mask_path), 0)
                mask = cv2.resize(mask, (img_size, img_size))
                mask = (mask > 127).astype(np.float32)

                self.images.append(img)
                self.masks.append(mask)

        if len(self.images) == 0:
            raise ValueError(f"No matching pairs found in {images_dir}. Verify file names.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Pull from RAM and convert to Tensor instantly
        img = torch.from_numpy(self.images[idx]).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(self.masks[idx]).unsqueeze(0)
        return img, mask

# Initialize with verified local paths
dataset = FastDataset(LOCAL_IMG_DIR, LOCAL_MASK_DIR)

# High-speed DataLoader configuration
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0, # 0 is faster when data is already in RAM
    pin_memory=True # Speeds up transfer to GPU
)

print("🚀 Ready to train DeepLabv3+ at maximum speed!")

⚡ Pre-loading 11827 images into RAM for 10x speed...


100%|██████████| 11827/11827 [00:01<00:00, 7824.26it/s]

🚀 Ready to train DeepLabv3+ at maximum speed!


In [34]:
import os
from pathlib import Path

# 1. Double check the directories
IMAGE_PATH = '/content/local_images'
MASK_PATH = '/content/local_masks'

# If these are empty, the previous copy step might have failed
# due to a slight difference in your Google Drive folder structure.
images_found = list(Path(IMAGE_PATH).glob('*.jpg'))
masks_found = list(Path(MASK_PATH).glob('*.png'))

print(f"Total JPGs found: {len(images_found)}")
print(f"Total Masks found: {len(masks_found)}")

# 2. Robust Dataset Class
class SimpleDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])

        # This list comprehension must find matches to avoid the ValueError
        all_imgs = list(self.images_dir.glob('*.jpg'))
        self.pairs = []

        for img in all_imgs:
            # Check for both "name_mask.png" and "name.png" just in case
            mask_option1 = self.masks_dir / f"{img.stem}_mask.png"
            mask_option2 = self.masks_dir / f"{img.stem}.png"

            if mask_option1.exists():
                self.pairs.append((img, mask_option1))
            elif mask_option2.exists():
                self.pairs.append((img, mask_option2))

        print(f"✅ Successfully matched {len(self.pairs)} image-mask pairs.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = cv2.imread(str(img_path))[...,::-1]
        mask = cv2.imread(str(mask_path), 0)
        mask = (mask > 127).astype(np.float32)
        return self.transform(img), self.transform(mask)[0][None]

# 3. Initialize with the updated logic
dataset = SimpleDataset(IMAGE_PATH, MASK_PATH)

if len(dataset) == 0:
    print("❌ ERROR: Still no pairs found. Check if your Drive folder names are correct.")
else:
    train_loader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=2)
    print("🚀 DataLoader ready to roll!")

Total JPGs found: 11827
Total Masks found: 2135
✅ Successfully matched 198 image-mask pairs.
🚀 DataLoader ready to roll!


In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models
import torchvision.transforms as transforms
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Constants - Reduced Epochs for faster completion
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 5  # Reduced as we are using a pretrained backbone
LR = 0.0005
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimpleDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])
        imgs = list(self.images_dir.glob('*.jpg'))
        self.pairs = [img for img in imgs if (self.masks_dir / f"{img.stem}_mask.png").exists()]

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_path = self.pairs[idx]
        mask_path = self.masks_dir / f"{img_path.stem}_mask.png"
        img = cv2.imread(str(img_path))[...,::-1]
        mask = cv2.imread(str(mask_path), 0)
        mask = (mask > 127).astype(np.float32)
        return self.transform(img), self.transform(mask)[0][None]

# Faster Model: DeepLabV3 with MobileNetV3-Large Backbone
def get_fast_deeplab(output_channels=1):
    model = models.segmentation.deeplabv3_mobilenet_v3_large(weights='DEFAULT')
    # Adjust the classifier head for 1 channel output
    model.classifier[4] = nn.Conv2d(256, output_channels, 1)
    return model.to(DEVICE)

# Initialize Data using LOCAL paths
dataset = SimpleDataset('/content/local_images', '/content/local_masks')
train_loader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=4) # Increased workers

model = get_fast_deeplab()
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

print(f"🚀 Training Lightweight DeepLabv3+ on {DEVICE}")

for epoch in range(EPOCHS):
    model.train()
    loss_total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for imgs, masks in pbar:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)['out']
        preds = torch.sigmoid(outputs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        loss_total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

torch.save(model.state_dict(), 'deeplabv3_fast.pth')
print("✅ Saved Fast Model!")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Downloading: "https://download.pytorch.org/models/deeplabv3_mobilenet_v3_large-fc3c493d.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_mobilenet_v3_large-fc3c493d.pth


100%|██████████| 42.3M/42.3M [00:00<00:00, 89.1MB/s]


🚀 Training Lightweight DeepLabv3+ on cpu


Epoch 5: 100%|██████████| 7/7 [00:20<00:00,  2.92s/it, loss=0.3390]


✅ Saved Fast Model!


In [36]:
def save_final_result(img_path):
    model = get_fast_deeplab()
    model.load_state_dict(torch.load('deeplabv3_fast.pth'))
    model.eval()

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Preprocess
    input_tensor = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])(img[...,::-1]).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(input_tensor)['out']
        mask = (torch.sigmoid(output)[0,0].cpu().numpy() > 0.5).astype(np.uint8)
        mask = cv2.resize(mask, (w, h))

    result = img * mask[:,:,None]

    # Target folder on Drive
    drive_output = '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/output/deeplabv3_result.jpg'
    cv2.imwrite(drive_output, result)
    print(f"🎯 Output saved to Google Drive: {drive_output}")

# Execute
save_final_result("/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/train2017/000000532933.jpg")

🎯 Output saved to Google Drive: /content/drive/MyDrive/AI_Vision_Extract_Nov25/data/output/deeplabv3_result.jpg
